# Продвинутые модели (CatBoost)

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project():
    for p in [Path.cwd(), Path.cwd().parent]:
        if (p / "src" / "models" / "train.py").exists():
            return p
    raise FileNotFoundError("Запускайте ноутбук из project/ или project/notebooks/")

PROJECT = _find_project()
sys.path.insert(0, str(PROJECT))
from src.data.notebook_helpers import get_project_paths

project_dir, data_path, plots_dir = get_project_paths()

## 1. Подготовка данных

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.data.notebook_helpers import load_sample, add_kmeans_cluster, get_xy

df = load_sample(100000, random_state=42)
df, _ = add_kmeans_cluster(df)
X, y = get_xy(df, with_cluster=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Bagging Regressor

In [ ]:
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=10,
    random_state=42
)
bagging.fit(X_train, y_train)

y_pred_bag = bagging.predict(X_test)

mae_bag = mean_absolute_error(y_test, y_pred_bag)
rmse_bag = np.sqrt(mean_squared_error(y_test, y_pred_bag))
r2_bag = r2_score(y_test, y_pred_bag)

print(f"Bagging Regressor:")
print(f"  MAE: {mae_bag:.0f}")
print(f"  RMSE: {rmse_bag:.0f}")
print(f"  R2: {r2_bag:.3f}")

## 3. CatBoost Regressor

In [ ]:
catboost = CatBoostRegressor(
    iterations=400,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=False
)
catboost.fit(X_train, y_train)

y_pred_cb = catboost.predict(X_test)

mae_cb = mean_absolute_error(y_test, y_pred_cb)
rmse_cb = np.sqrt(mean_squared_error(y_test, y_pred_cb))
r2_cb = r2_score(y_test, y_pred_cb)

print(f"CatBoost:")
print(f"  MAE: {mae_cb:.0f}")
print(f"  RMSE: {rmse_cb:.0f}")
print(f"  R2: {r2_cb:.3f}")

## 4. Важность признаков

In [ ]:
feature_importance = catboost.get_feature_importance()
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importance)
plt.xlabel('Важность')
plt.title('Важность признаков (CatBoost)')
plt.savefig(os.path.join(plots_dir, '04_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()